# Publication summary of $-F_A(Q^2)$

This notebook turns selected joint PROfit posterior chains into median $-F_A$ curves and pointwise 68% credible bands. Edit only the **Figure controls** cell for normal use, then run all cells. The gray band is the one selected prior; the lower panel shows every result relative to that prior's median.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt

start = Path.cwd().resolve()
NOTEBOOK_DIR = next(
    (path / 'python' / 'scripts' for path in (start, *start.parents)
     if (path / 'python' / 'scripts' / 'postfit_physical_parameters.py').is_file()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError('Could not locate python/scripts/postfit_physical_parameters.py')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from postfit_physical_parameters import SPECS, load_fit, plot_fa_summary


## Figure controls

Set a fit to `True` to draw its posterior. `COMPARISON_PRIOR` must be one key from the same list, or `None` for the conventional $M_A=1.014$ GeV dipole denominator. The comparison prior does not need to be enabled as a posterior. `LEGEND_LABELS` overrides individual legend entries; by default the prior reads "MINERvA (2026) prior, $k_{\max}=6$" and the posterior "Posterior, $k_{\max}=6$", naming its prior only when several priors appear.

In [ ]:
# Data selection
SUITE = 'nuwro_fit_results'  # NuWro: nuwro_fit_results; Asimov: asimov_fit_results
SHOW_POSTFIT = {
    'minerva_k8': False,
    'minerva_k7': False,
    'minerva_k6': True,
    'lqcd_k6': False,
    'lqcd_k7': False,
    'minerva_lqcd_k6': False,
    'minerva_lqcd_k7': False,
    "minerva_k6_uniform": False,
    "minerva_k6_uniform_nuisance": False,
}
COMPARISON_PRIOR = 'minerva_k6'  # one key above, or None

# Optional legend overrides. Posteriors are keyed by fit key and the comparison
# prior by '<key> prior'; anything not listed keeps the standard text, e.g.
# 'MINERvA (2026) prior, $k_{\max}=6$' and 'Posterior, $k_{\max}=6$'.
LEGEND_LABELS = {}

# Chain and plot controls
BURN_IN = 0
THIN = 1
Q2_RANGE = (0.01, 2.0)
X_SCALE = 'log'  # 'log' or 'linear'
RATIO_ZOOM = (0.01, 0.25, 0.90, 1.10)  # (Q2 min, Q2 max, ratio min, ratio max), or None
MAX_CURVES = 50_000

# Publication output
SAVE_FIGURE = True
OUTPUT_STEM = NOTEBOOK_DIR.parents[1] / 'figs' / SUITE / 'fa_postfit_summary'
SAVE_FORMATS = ('pdf',)
PNG_DPI = 600

In [ ]:
selected = [key for key, enabled in SHOW_POSTFIT.items() if enabled]
requested = set(selected) | ({COMPARISON_PRIOR} if COMPARISON_PRIOR else set())
specs = {spec.key: spec for spec in SPECS}
unknown = requested - set(specs)
if unknown:
    raise ValueError(f'Unknown fit keys: {sorted(unknown)}')
if not requested:
    raise ValueError('Enable at least one posterior or choose a comparison prior.')

results = {}
for key in requested:
    result = load_fit(specs[key], SUITE, burn_in=BURN_IN, thin=THIN)
    if result is None:
        raise FileNotFoundError(f'No unique PROfile ROOT file for {key!r} in {SUITE!r}')
    results[key] = result
    print(f'{key}: {len(result["samples"]):,} posterior samples')

In [ ]:
figure = plot_fa_summary(
    results,
    show=selected,
    comparison_prior=COMPARISON_PRIOR,
    q2_range=Q2_RANGE,
    max_samples=MAX_CURVES,
    ratio_zoom=RATIO_ZOOM,
    xscale=X_SCALE,
    labels=LEGEND_LABELS,
)
display(figure)

if SAVE_FIGURE:
    OUTPUT_STEM.parent.mkdir(parents=True, exist_ok=True)
    for extension in SAVE_FORMATS:
        path = OUTPUT_STEM.with_suffix('.' + extension.lstrip('.'))
        kwargs = dict(bbox_inches='tight', pad_inches=.03, facecolor='white')
        if extension.lower().lstrip('.') == 'png':
            kwargs['dpi'] = PNG_DPI
        figure.savefig(path, **kwargs)
        print(f'Saved {path}')
plt.close(figure)

## Open-data summary

Reproduce the summary above with the open-data posterior chains and the same fit selection and plotting controls.

In [ ]:
OPEN_DATA_SUITE = 'opendata_fit_results'

open_data_results = {}
for key in requested:
    result = load_fit(specs[key], OPEN_DATA_SUITE, burn_in=BURN_IN, thin=THIN)
    if result is None:
        raise FileNotFoundError(
            f'No unique PROfile ROOT file for {key!r} in {OPEN_DATA_SUITE!r}'
        )
    open_data_results[key] = result
    print(f'{key}: {len(result["samples"]):,} posterior samples')

open_data_figure = plot_fa_summary(
    open_data_results,
    show=selected,
    comparison_prior=COMPARISON_PRIOR,
    q2_range=Q2_RANGE,
    max_samples=MAX_CURVES,
    ratio_zoom=RATIO_ZOOM,
    xscale=X_SCALE,
    labels=LEGEND_LABELS,
)
display(open_data_figure)

if SAVE_FIGURE:
    open_data_output_stem = (
        NOTEBOOK_DIR.parents[1] / 'figs' / OPEN_DATA_SUITE / 'fa_postfit_summary'
    )
    open_data_output_stem.parent.mkdir(parents=True, exist_ok=True)
    for extension in SAVE_FORMATS:
        path = open_data_output_stem.with_suffix('.' + extension.lstrip('.'))
        kwargs = dict(bbox_inches='tight', pad_inches=.03, facecolor='white')
        if extension.lower().lstrip('.') == 'png':
            kwargs['dpi'] = PNG_DPI
        open_data_figure.savefig(path, **kwargs)
        print(f'Saved {path}')
plt.close(open_data_figure)

## Asimov-data summary

Reproduce the summary above with the Asimov posterior chains and the same fit selection and plotting controls.

In [ ]:
ASIMOV_DATA_SUITE = 'asimov_fit_results'

asimov_data_results = {}
for key in requested:
    result = load_fit(specs[key], ASIMOV_DATA_SUITE, burn_in=BURN_IN, thin=THIN)
    if result is None:
        raise FileNotFoundError(
            f'No unique PROfile ROOT file for {key!r} in {ASIMOV_DATA_SUITE!r}'
        )
    asimov_data_results[key] = result
    print(f'{key}: {len(result["samples"]):,} posterior samples')

asimov_data_figure = plot_fa_summary(
    asimov_data_results,
    show=selected,
    comparison_prior=COMPARISON_PRIOR,
    q2_range=Q2_RANGE,
    max_samples=MAX_CURVES,
    ratio_zoom=RATIO_ZOOM,
    xscale=X_SCALE,
    labels=LEGEND_LABELS,
)
display(asimov_data_figure)

if SAVE_FIGURE:
    asimov_data_output_stem = (
        NOTEBOOK_DIR.parents[1] / 'figs' / ASIMOV_DATA_SUITE / 'fa_postfit_summary'
    )
    asimov_data_output_stem.parent.mkdir(parents=True, exist_ok=True)
    for extension in SAVE_FORMATS:
        path = asimov_data_output_stem.with_suffix('.' + extension.lstrip('.'))
        kwargs = dict(bbox_inches='tight', pad_inches=.03, facecolor='white')
        if extension.lower().lstrip('.') == 'png':
            kwargs['dpi'] = PNG_DPI
        asimov_data_figure.savefig(path, **kwargs)
        print(f'Saved {path}')
plt.close(asimov_data_figure)

## Uniform-prior posterior against LQCD

Supports the discussion of the uniform-prior $z$-expansion fit: $-F_A$ for the $k_{\max}=6$ uniform-prior posterior against the LQCD (and, for context, MINERvA) prior, the ratio to the LQCD median, and the pointwise displacement $\Delta/\sigma_{\rm post}=(\text{posterior median}-\text{LQCD median})/\sigma_{\rm post}$, with $\sigma_{\rm post}$ the half-width of the posterior 68% interval. The shaded window is the $Q^2$ range quoted in the text; the printed numbers are the ones the text cites.

In [ ]:
from postfit_physical_parameters import plot_fa_tension

TENSION_SUITE = 'opendata_fit_results'
TENSION_WINDOW = (0.05, 0.5)  # GeV^2, the range quoted in the text
TENSION_AGREEMENT_Q2 = 0.8  # GeV^2, lower edge of the quoted agreement region

tension_results = {
    key: load_fit(specs[key], TENSION_SUITE, burn_in=BURN_IN, thin=THIN)
    for key in ('minerva_k6_uniform', 'lqcd_k6', 'minerva_k6')
}
tension_figure, tension_table = plot_fa_tension(
    tension_results, highlight=TENSION_WINDOW, max_samples=MAX_CURVES,
)
display(tension_figure)

window = tension_table.query('@TENSION_WINDOW[0] <= Q2 <= @TENSION_WINDOW[1]')
high = tension_table.query('Q2 >= @TENSION_AGREEMENT_Q2')
print(f'{TENSION_WINDOW[0]} <= Q2 <= {TENSION_WINDOW[1]} GeV^2: '
      f'|F_A| ratio {window.ratio.min():.3f}-{window.ratio.max():.3f}, '
      f'displacement {window.pull.min():.2f}-{window.pull.max():.2f} sigma_post')
print(f'Q2 >= {TENSION_AGREEMENT_Q2} GeV^2: ratio {high.ratio.min():.3f}-{high.ratio.max():.3f}, '
      f'displacement {high.pull.min():.2f} to {high.pull.max():.2f} sigma_post')

if SAVE_FIGURE:
    tension_stem = NOTEBOOK_DIR.parents[1] / 'figs' / TENSION_SUITE / 'fa_uniform_vs_lqcd'
    tension_stem.parent.mkdir(parents=True, exist_ok=True)
    for extension in SAVE_FORMATS:
        path = tension_stem.with_suffix('.' + extension.lstrip('.'))
        tension_figure.savefig(path, bbox_inches='tight', pad_inches=.03,
                               facecolor='white', dpi=PNG_DPI)
        print(f'Saved {path}')
plt.close(tension_figure)

## NuWro fake-data posteriors against the NuWro input form factor

Supports the supplemental discussion that the enhanced $|F_A|$ on the NuWro fake data does not come from the sample's axial form factor. The NuWro sample is generated with a dipole, $M_A=1.03$ GeV (black, same $F_A(0)$ as the dipole fits). Only the post-fit $F_A$ is drawn: the dipole fit with the Gaussian $M_A$ prior, the uniform-prior $z$-expansion with fitted nuisances, and the LQCD-prior $z$-expansion. Every fit is drawn relative to that input: ratio in the middle panel, $(\text{posterior median}-\text{input})/\sigma_{\rm post}$ in the bottom panel, with $\sigma_{\rm post}$ the half-width of the posterior 68% interval. For the dipole fits the $F_A$-space displacement exceeds the $M_A$-space one, $(M_A^{\rm med}-1.03)/\sigma_{M_A}$, because $F_A$ is concave in $M_A$; both are printed. The shaded window is the $Q^2$ range quoted in the text.

In [ ]:
import numpy as np
from postfit_physical_parameters import NUWRO_INPUT_FITS, NUWRO_INPUT_MA, plot_fa_vs_nuwro_input

INPUT_SUITE = 'nuwro_fit_results'
INPUT_WINDOW = (0.05, 0.5)  # GeV^2, the range quoted in the text

input_results = {
    key: load_fit(specs[key], INPUT_SUITE, burn_in=BURN_IN, thin=THIN)
    for key in [fit[0] for fit in NUWRO_INPUT_FITS]
}
input_figure, input_table = plot_fa_vs_nuwro_input(
    input_results, highlight=INPUT_WINDOW, max_samples=MAX_CURVES,
)
display(input_figure)

window = input_table.query('@INPUT_WINDOW[0] <= Q2 <= @INPUT_WINDOW[1]')
print(f'{INPUT_WINDOW[0]} <= Q2 <= {INPUT_WINDOW[1]} GeV^2, relative to the NuWro input:')
display(window.groupby('fit', sort=False).agg(
    ratio_min=('ratio', 'min'), ratio_max=('ratio', 'max'),
    pull_min=('pull', 'min'), pull_max=('pull', 'max'),
).round(3))
for key in [k for k in input_results if specs[k].prior is None]:
    ma = input_results[key]['samples'][:, 0]
    q16, q50, q84 = np.quantile(ma, [.16, .50, .84])
    print(f'{key}: M_A = {q50:.3f} +{q84 - q50:.3f} -{q50 - q16:.3f} GeV, '
          f'(M_A - {NUWRO_INPUT_MA}) / sigma = {(q50 - NUWRO_INPUT_MA) / ((q84 - q16) / 2):.2f}')

if SAVE_FIGURE:
    input_stem = NOTEBOOK_DIR.parents[1] / 'figs' / INPUT_SUITE / 'fa_vs_nuwro_input'
    input_stem.parent.mkdir(parents=True, exist_ok=True)
    for extension in SAVE_FORMATS:
        path = input_stem.with_suffix('.' + extension.lstrip('.'))
        input_figure.savefig(path, bbox_inches='tight', pad_inches=.03,
                             facecolor='white', dpi=PNG_DPI)
        print(f'Saved {path}')
plt.close(input_figure)
